In [ ]:
import sys,os
notebook_dir = os.getcwd()  # Gets current working directory
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.append(parent_dir)
sys.path.append(parent_dir + "/python_code")

In [ ]:
import sys
import time
import numpy as np
import scipy.sparse as sp

import matplotlib.pyplot as plt

from scipy.sparse import coo_matrix, csc_matrix, csr_matrix, lil_matrix
from sksparse.cholmod import cholesky, cholesky_AAt
from pykdtree.kdtree import KDTree

from implementation.Cloth import Cloth
from implementation.utils import createRectangularMesh

np.set_printoptions(threshold=sys.maxsize, suppress=True, linewidth=140)

In [ ]:
X, T = createRectangularMesh(a=2, b=1, na=3, nb=2, h=0.5)
verts, faces = createRectangularMesh(a=2, b=1, na=3, nb=2, h=0.5)
print(f'\n verts = {verts}')
print(f'\n faces = {faces}')
# verts[:,2] += 0.7; 
# verts += 0.0001*np.random.randn(verts.shape[0],3) 

In [ ]:
positions = np.array(verts, order='F')
velocities = np.zeros(positions.shape, order='F') + 1e-12
# order F is Fortran/column-major storage. 
# This matters because later the code often flattens arrays with:

In [ ]:
positions[:,1]

In [ ]:
fig = plt.figure(figsize=(8,4))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(positions[:,0], positions[:,1], positions[:,2], s=30)

for i,p in enumerate(positions):
    ax.text(p[0], p[1], p[2] + 0.005, i)

for f in faces:
    cyclic = list(f) + [f[0]]
    quad = positions[cyclic]
    # quad = quad.append(quad[0])
    ax.plot(quad[:,0], quad[:,1], quad[:,2], '-k')

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("Initial quad mesh")

In [ ]:
# all x coordinates, then y and then z
phi0 = positions.reshape((3 * positions.shape[0],), order='F')
phi0

In [ ]:
# For a 4-node quad element,
# shape functions help with the computation of points on the surface
# If I pull the 4 corners to 4 positions in 3D, where does the material point at 30% across and 70% up go?
# So a reference square [-1, 1] * [-1, 1] is used and then mapped to each surface.
# 2x2 Gauss quadrature points for a bilinear quad
w = np.array([1, 1, 1, 1], dtype=float)

nodesCoord = np.array([
    [-1, -1],
    [ 1, -1],
    [ 1,  1],
    [-1,  1]
], dtype=float)

z = nodesCoord / np.sqrt(3) #FEM element uses 2-point Gauss quadrature in each direction, 
# whose sampling points on [−1,1]  are exactly ±1/sqrt(3)
xi = z[:, 0]
eta = z[:, 1]

# [None, :] adds a new dimension from (,3) to (1, 3)
N = (1/4) * np.block([
    [((1-xi)*(1-eta))[None, :]],
    [((1+xi)*(1-eta))[None, :]],
    [((1+xi)*(1+eta))[None, :]],
    [((1-xi)*(1+eta))[None, :]]
])

# derivatives of the shape functions with respect to eta and xi
# They are needed to compute:local tangents, metric coefficients, area scaling, gradient-related quantities.

Nxi = (1/4) * np.block([
    [(eta - 1)[None, :]],
    [(1 - eta)[None, :]],
    [(1 + eta)[None, :]],
    [(-1 - eta)[None, :]]
]).T

Neta = (1/4) * np.block([
    [(xi - 1)[None, :]],
    [(-1 - xi)[None, :]],
    [(1 + xi)[None, :]],
    [(1 - xi)[None, :]]
]).T

print("Gauss points z =")
print(z)

# print("\nShape function values N (rows=node, cols=quadrature point) =")
# print(N)

# print("\nNxi =")
# print(Nxi)

# print("\nNeta =")
# print(Neta)

In [ ]:
face0 = faces[0]
X_face0 = np.vstack([positions[node] for node in face0])
X_face0

In [ ]:
# Center point of the first face: eta = xi = 0. Thus divide by 4 and add the four corners or take mean
phi00 = X_face0.mean(axis=0) # axis=0 meaning operate on rows


In [ ]:
# # compute all unique edges of the quad mesh
# edges = []
# container = []
# for f in faces:
#     edges.append([[f[0], f[1]], [f[1], f[2]], [f[2], f[3]], [f[3], f[0]]]) 

# for edge in edges:
#     for i in range(4):
#         container.append(tuple(sorted([edge[i][0], edge[i][1]])))

# unique_edges = np.array(sorted(set(container)))
# unique_edges

# Better way (avoiding loops)

faces_arr = np.array(faces, dtype=int)

e1 = faces_arr[:, [0, 1]]
e2 = faces_arr[:, [1, 2]]
e3 = faces_arr[:, [2, 3]]
e4 = faces_arr[:, [3, 0]]

edges = set(map(frozenset, e1))
edges.update(set(map(frozenset, e2)))
edges.update(set(map(frozenset, e3)))
edges.update(set(map(frozenset, e4)))

edges = list(edges)
edges_matrix = np.array(list(map(list, edges)), dtype=int)
n_edges = len(edges)

print("n_edges =", n_edges)
print(edges_matrix)

# edges_sorted = np.sort(edges_matrix, axis=1)
# edges_sorted = edges_sorted[np.lexsort((edges_sorted[:,1], edges_sorted[:,0]))]

# print(edges_sorted)

In [ ]:
# # build the edge-to-vertex adjacency matrix A0
# # entry (i, j) = 1 if vertex j is part of edge i

# n_verts = verts.shape[0]
# n_edges = edges_matrix.shape[0]
# A0 = np.zeros((n_edges, n_verts))
# for i in range(n_edges):
#     # for j in range(n_verts):
#     #     if j in unique_edges[i]:
#     #         A0[i,j] = 1
#     a, b = edges_matrix[i]
#     A0[i, a] = 1
#     A0[i, b] = 1
# A0

# Better way (avoiding loops)
# For example: edges [0, 1], [1, 4], [3, 4]
# row: [0 1 2 0 1 2], col = [0, 1, 3, 1, 4, 4]. Thus ones in the matrix are exactly at
# [[0, 0], [1, 1], ..., [2, 4]]
n_verts = len(verts)
row = np.arange(n_edges)
row = np.concatenate((row, row))

col = np.concatenate((edges_matrix[:, 0], edges_matrix[:, 1]))
data = np.ones_like(row)

A0 = coo_matrix((data, (row, col)), shape=(n_edges, n_verts)).tocsr()
# A0t = A0.T.tocsr()

print("A0 shape:", A0.shape)
print(A0.toarray())


In [ ]:
# Adjacency matrix A1: faces and edges

ind_edges = dict((k, i) for i, k in enumerate(edges))
n_faces = len(faces)

row = np.arange(n_faces)
row = np.concatenate((row, row, row, row))

se1 = list(map(frozenset, faces_arr[:, [0, 1]]))
se2 = list(map(frozenset, faces_arr[:, [1, 2]]))
se3 = list(map(frozenset, faces_arr[:, [2, 3]]))
se4 = list(map(frozenset, faces_arr[:, [3, 0]]))

ind1 = np.array([ind_edges[x] for x in se1])
ind2 = np.array([ind_edges[x] for x in se2])
ind3 = np.array([ind_edges[x] for x in se3])
ind4 = np.array([ind_edges[x] for x in se4])

col = np.concatenate((ind1, ind2, ind3, ind4))
data = np.ones_like(row)

A1 = coo_matrix((data, (row, col)), shape=(n_faces, n_edges)).tocsr()

print("A1 shape:", A1.shape)
print(A1.toarray())

In [ ]:
# Adjacency matrix A2: faces and vertices
row = np.arange(n_faces)
row = np.concatenate((row, row, row, row))

col = np.concatenate((faces_arr[:,0], faces_arr[:,1], faces_arr[:,2], faces_arr[:,3]))
data = np.ones_like(row)

A2 = coo_matrix((data, (row, col)), shape=(n_faces, n_verts)).tocsr()
A2t = A2.T.tocsr()

nodes_faces_count = np.array(A2.sum(axis=0)).ravel()
Am = sp.vstack([sp.eye(n_verts), 0.25 * A2]).tocsr() 
# is the matrix that appends one center per face for rendering.

print("A2 shape:", A2.shape)
print(A2.toarray())

# print("\nnodes_faces_count:")
# print(nodes_faces_count)

# print("\nAm shape:", Am.shape)
# print(Am.toarray())

In [ ]:
# calculate boundary nodes and edges
sumCols = np.array(A1.T.sum(axis=1)).ravel()
boundary_edge_idx = np.where(sumCols == 1)[0]

edges_bnd = edges_matrix[boundary_edge_idx, :]
nodes_bnd = np.unique(edges_bnd.reshape(-1))

print("\nboundary edges:")
print(edges_bnd)

print("\nboundary nodes:")
print(nodes_bnd)


In [ ]:
# 13. triangulate the quad mesh for display/smoothing
k1, k2, k3, k4 = faces_arr[:, 0], faces_arr[:, 1], faces_arr[:, 2], faces_arr[:, 3]
n_tot = n_verts + n_faces
k5 = np.arange(n_verts, n_tot)

triangles = np.vstack([
    np.column_stack([k1, k2, k5]),
    np.column_stack([k2, k3, k5]),
    np.column_stack([k3, k4, k5]),
    np.column_stack([k4, k1, k5]),
])

print("triangles:")
print(triangles)

# compute triangle edges

edges_tri = np.vstack([
    triangles[:, [0, 1]],
    triangles[:, [1, 2]],
    triangles[:, [2, 0]]
])

edges_tri = np.sort(edges_tri, axis=1)
edges_tri = np.unique(edges_tri, axis=0)

# print("edges_tri:")
# print(edges_tri)

In [ ]:
# 14. Smoothing matrix S (for visualization)
S = sp.lil_matrix((n_tot, n_tot))
alpha = 0.75

for n in range(n_tot):
    aux = (edges_tri[:, 0] == n) | (edges_tri[:, 1] == n)
    edges_n = edges_tri[aux]
    neighs_n = np.setdiff1d(np.unique(edges_n), n)

    if n in nodes_bnd:
        S[n, n] = 1
    else:
        S[n, n] = alpha
        if len(neighs_n) > 0:
            S[n, neighs_n] = (1 - alpha) / len(neighs_n)

print("S shape:", S.shape)
print(S.toarray())

In [ ]:
# 15. Helper functions used later
def innerProduct(u, v):
    return np.einsum('ij,ij->i', u, v)

def computeNorm(w):
    return np.sqrt(innerProduct(w, w))

def normalize(w):
    nrm = computeNorm(w) + 1e-12
    return w / nrm[:, None]

# Quick test
test = np.array([[3, 4, 0], [1, 2, 2]], dtype=float)
print(computeNorm(test))
print(normalize(test))

In [ ]:
# 16. Standard FEM assymbly of blocks: Mass and Laplacian matrices 
M = sp.lil_array((n_verts, n_verts), dtype=float)
L = sp.lil_array((n_verts, n_verts), dtype=float)
# lil_array is a list of lists for storing sparse matrices in an efficient way
# instead of storing [0, 0, 0, 4.2, 0, 0, 1.7, 0], it just stores columns = [3, 6]; values = [4.2, 1.7]

# same "mat1" as in the class
mat1 = [np.kron(N[j:j+1].T, N[j:j+1]) for j in range(4)] # mass-matrix integrand

for i in range(n_faces):
    X_i = np.vstack([positions[node] for node in faces_arr[i]])   # shape (4,3)

    Me = np.zeros((4, 4), dtype=float)
    Le = np.zeros((4, 4), dtype=float)

    for j in range(4):
        # tangent vectors of the surface patch
        phi_xi  = Nxi[j]  @ X_i     # shape (3,); @ is matric multiplication
        phi_eta = Neta[j] @ X_i     # shape (3,)

        dphi = np.vstack([phi_xi, phi_eta])   # shape (2,3)

        E = phi_xi @ phi_xi.T # stretch along xi
        F = phi_xi @ phi_eta.T # stretch along eta
        G = phi_eta @ phi_eta.T # shear between xi and eta directions

        m = np.array([[E, F], [F, G]], dtype=float)
        dS = np.sqrt(abs(E * G - F**2)) * w[j]

        rhs = np.vstack([Nxi[j], Neta[j]])   # shape (2,4)
        Nxyz_k = dphi.T @ np.linalg.solve(m, rhs)   # shape (3,4)

        Me += mat1[j] * dS
        # Le below represents: (Dell N)^T (Dell N): Term in Laplacian/stiffness matrices
        Le += (Nxyz_k[0:1].T @ Nxyz_k[0:1] +
               Nxyz_k[1:2].T @ Nxyz_k[1:2] +
               Nxyz_k[2:3].T @ Nxyz_k[2:3]) * dS 
    for j in range(4):
        for k in range(4):
            M[faces_arr[i, j], faces_arr[i, k]] += Me[j, k]
            L[faces_arr[i, j], faces_arr[i, k]] += Le[j, k]

M = M.tocsc()
L = L.tocsc()
# csc better for arithmetic, factorization, later sparse solves

print("M shape:", M.shape)
print(M.toarray())

print("\nL shape:", L.shape)
print(L.toarray())

In [ ]:
# 17. Lump the mass matrix
m_lum = np.array(M.sum(axis=1)).ravel()
m_inv = 1.0 / m_lum
m_sqrt = 1.0 / np.sqrt(m_lum)

M_lum = sp.diags(m_lum).tocsc()
M_inv = sp.diags(m_inv).tocsc()

print("m_lum =", m_lum)
print("m_inv =", m_inv)
print("m_sqrt =", m_sqrt)

print("\nM_lum =")
print(M_lum.toarray())

In [ ]:
# 18. Build the stiffness matrix: K = L^T M_L^{-1} L
K = L.T @ M_inv @ L

print("K shape:", K.shape)
print(K.toarray())

In [ ]:
# 19. expand the mass matrix into xyz form
M_xyz = sp.block_diag((M_lum, M_lum, M_lum)).tocsc()

m_sqrt_xyz = np.concatenate([m_sqrt, m_sqrt, m_sqrt])
m_sqrt_xyz_col = m_sqrt_xyz.reshape((-1,), order='F')

# print("M_xyz shape:", M_xyz.shape)
# print(M_xyz.toarray())

# print("\nm_sqrt_xyz =")
# print(m_sqrt_xyz)

In [ ]:
# 20. Gravity vector
Fg = sp.lil_matrix((n_verts, 3), dtype=float)
Fg[:, 2] = -m_lum
Fg = Fg.tocsc()

print("Fg =")
print(Fg.toarray())

In [ ]:
# 21. compute stretch bars
# The paper uses edge-length constraints on boundary curves and metric constraints in the interior; 
# this code uses a practical bar-based stretch constraint.
bars = np.vstack([
    faces_arr[:, [0, 1]],
    faces_arr[:, [1, 2]],
    faces_arr[:, [2, 3]],
    faces_arr[:, [3, 0]]
])

bars = np.unique(np.sort(bars, axis=1), axis=0)

print("bars:")
print(bars)

In [ ]:
# 22. For each bar ((i,j)), the rest value is: |p_j-p_i|^2

bars0 = bars[:, 0]
bars1 = bars[:, 1]

vec_rest = positions[bars1] - positions[bars0]
stretch_val0 = np.einsum('ij,ij->i', vec_rest, vec_rest)

print("rest edge vectors:")
print(vec_rest)

print("\nrest squared lengths:")
print(stretch_val0)

In [ ]:
# 23. Shear neighbour structure: Easy to understand with the plot of the quads above
# neighbour nodes where shear can be applied
neighs_xi = {i: set() for i in range(n_verts)}
neighs_eta = {i: set() for i in range(n_verts)}

for face in faces_arr:
    # xi direction
    neighs_xi[face[0]].add(face[1])
    neighs_xi[face[1]].add(face[0])
    neighs_xi[face[2]].add(face[3])
    neighs_xi[face[3]].add(face[2])

    # eta direction
    neighs_eta[face[0]].add(face[3])
    neighs_eta[face[3]].add(face[0])
    neighs_eta[face[1]].add(face[2])
    neighs_eta[face[2]].add(face[1])

print("neighs_xi:")
for k, v in neighs_xi.items():
    print(k, sorted(list(v)))

print("\nneighs_eta:")
for k, v in neighs_eta.items():
    print(k, sorted(list(v)))

In [ ]:
# 24. Build shear tuples
# This is the discrete version of preserving the metric’s mixed term 
# (F=\langle \phi_\xi,\phi_\eta \rangle), including special corner handling.
neighs_shear = []
corners_shear = []

for n in range(n_verts):
    if len(neighs_xi[n]) == 2 and len(neighs_eta[n]) == 2:
        neighs_shear.append(list(neighs_xi[n]) + list(neighs_eta[n]))
    elif len(neighs_xi[n]) == 2 and len(neighs_eta[n]) == 1:
        neighs_shear.append([n] + list(neighs_eta[n]) + list(neighs_xi[n]))
    elif len(neighs_xi[n]) == 1 and len(neighs_eta[n]) == 2:
        neighs_shear.append([n] + list(neighs_xi[n]) + list(neighs_eta[n]))
    elif len(neighs_xi[n]) == 1 and len(neighs_eta[n]) == 1:
        corners_shear.append([n] + list(neighs_xi[n]) + [n] + list(neighs_eta[n]))

shear_neighs = np.array(neighs_shear, dtype=int) if len(neighs_shear) > 0 else np.zeros((0,4), dtype=int)
shear_corners = np.array(corners_shear, dtype=int) if len(corners_shear) > 0 else np.zeros((0,4), dtype=int)

print("shear_neighs:")
print(shear_neighs)

print("\nshear_corners:")
print(shear_corners)

In [ ]:
# 25. compute rest shear values
# For each shear tuple, the code evaluates a dot product 
# \langle \text{vec}_1,\text{vec}_2 \rangle: angle between them
# - if two local directions are orthogonal, the dot product is 0
# - if the patch skews, the dot product changes
all_shear = np.vstack([shear_neighs, shear_corners]) if (len(shear_neighs) + len(shear_corners)) > 0 else np.zeros((0,4), dtype=int)

if len(all_shear) > 0:
    neighs0 = all_shear[:, 0]
    neighs1 = all_shear[:, 1]
    neighs2 = all_shear[:, 2]
    neighs3 = all_shear[:, 3]

    vec1_rest = positions[neighs1] - positions[neighs0]
    vec2_rest = positions[neighs3] - positions[neighs2]
    shear_val0 = np.einsum('ij,ij->i', vec1_rest, vec2_rest)

    print("vec1_rest:")
    print(vec1_rest)

    print("\nvec2_rest:")
    print(vec2_rest)

    print("\nrest shear dot products:")
    print(shear_val0)
else:
    print("No shear tuples found.")

In [ ]:
# 26. Test deformation by hand
phi_test = positions.copy(order='F')

# perturb some nodes
phi_test[1] += np.array([0.10, 0.00, 0.05])
phi_test[4] += np.array([0.00, 0.05, -0.10])
phi_test[5] += np.array([-0.05, 0.00, 0.02])

print(phi_test)

In [ ]:
def plot_mesh_compare(positions, phi_test, edges_matrix, title="Mesh comparison"):
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')

    orig_color = 'tab:blue'
    def_color = 'tab:red'

    # Original edges
    for a, b in edges_matrix:
        ax.plot([positions[a, 0], positions[b, 0]],
                [positions[a, 1], positions[b, 1]],
                [positions[a, 2], positions[b, 2]],
                '--', linewidth=1, color=orig_color)

    # Deformed edges
    for a, b in edges_matrix:
        ax.plot([phi_test[a, 0], phi_test[b, 0]],
                [phi_test[a, 1], phi_test[b, 1]],
                [phi_test[a, 2], phi_test[b, 2]],
                linewidth=2, color=def_color)

    # Points
    ax.scatter(positions[:, 0], positions[:, 1], positions[:, 2],
               s=60, label='original', color=orig_color)
    ax.scatter(phi_test[:, 0], phi_test[:, 1], phi_test[:, 2],
               s=60, marker='^', label='deformed', color=def_color)

    for i in range(positions.shape[0]):
        ax.text(positions[i, 0], positions[i, 1], positions[i, 2], f'o{i}', fontsize=8)
        ax.text(phi_test[i, 0], phi_test[i, 1], phi_test[i, 2], f'd{i}', fontsize=8)

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.set_title(title)
    ax.legend()
    ax.view_init(elev=25, azim=-60)
    plt.show()

In [ ]:
plot_mesh_compare(positions, phi_test, edges_matrix, title="Test deformation")

In [ ]:
# 27. Evaluate stretch violation on the deformed mesh
vec_now = phi_test[bars1] - phi_test[bars0]
stretch_now = np.einsum('ij,ij->i', vec_now, vec_now)
stretch_violation = stretch_now - stretch_val0

print("current squared lengths:")
print(stretch_now)

print("\nstretch violation:")
print(stretch_violation)

In [ ]:
# 28. Evaluate shear violation on the deformed mesh
if len(all_shear) > 0:
    vec1_now = phi_test[neighs1] - phi_test[neighs0]
    vec2_now = phi_test[neighs3] - phi_test[neighs2]
    shear_now = np.einsum('ij,ij->i', vec1_now, vec2_now)
    shear_violation = shear_now - shear_val0

    print("current shear dot products:")
    print(shear_now)

    print("\nshear violation:")
    print(shear_violation)
else:
    print("No shear tuples found.")

In [ ]:
# 29. Inspect one strecth constraint by hand
i0, i1 = bars[0]
p0 = phi_test[i0]
p1 = phi_test[i1]

rest_sq = np.sum((positions[i1] - positions[i0])**2)
curr_sq = np.sum((p1 - p0)**2)
violation = curr_sq - rest_sq

print("bar:", (i0, i1))
print("rest_sq =", rest_sq)
print("curr_sq =", curr_sq)
print("violation =", violation)

In [ ]:
# 30. Build the stretch Jacobian
# Interpretation:
# - one row = one stretch constraint
# - nonzero columns = the xyz entries of the two endpoints involved in that edge

n_conds_str = bars.shape[0]

I = np.tile(np.arange(n_conds_str), 6)

J = np.concatenate([
    bars0,
    bars0 + n_verts,
    bars0 + 2*n_verts,
    bars1,
    bars1 + n_verts,
    bars1 + 2*n_verts
])

vec = phi_test[bars1] - phi_test[bars0]
grad1 = 2 * vec.flatten(order='F')
grad0 = -grad1

Kdata = np.concatenate([grad0, grad1])

grad_str = csc_matrix((Kdata, (I, J)), shape=(n_conds_str, 3*n_verts))
grad_str_T = grad_str.T.tocsr()

print("grad_str shape:", grad_str.shape)
print(grad_str.toarray())

In [ ]:
# 31. Shear Jacobian
if len(all_shear) > 0:
    n_crn = shear_corners.shape[0]
    n_conds_shr = all_shear.shape[0]

    In = np.tile(np.arange(shear_neighs.shape[0]), 12) if shear_neighs.shape[0] > 0 else np.array([], dtype=int)

    if shear_neighs.shape[0] > 0:
        v1 = shear_neighs[:, 0]
        v2 = shear_neighs[:, 1]
        v3 = shear_neighs[:, 2]
        v4 = shear_neighs[:, 3]

        Jn = np.concatenate([
            v1, v1 + n_verts, v1 + 2*n_verts,
            v2, v2 + n_verts, v2 + 2*n_verts,
            v3, v3 + n_verts, v3 + 2*n_verts,
            v4, v4 + n_verts, v4 + 2*n_verts
        ])
    else:
        Jn = np.array([], dtype=int)

    if n_crn > 0:
        Ic = np.tile(np.arange(n_crn), 9) + shear_neighs.shape[0]
        w1 = shear_corners[:, 0]
        w2 = shear_corners[:, 1]
        w3 = shear_corners[:, 3]

        Jc = np.concatenate([
            w1, w1 + n_verts, w1 + 2*n_verts,
            w2, w2 + n_verts, w2 + 2*n_verts,
            w3, w3 + n_verts, w3 + 2*n_verts
        ])

        I_shr = np.concatenate([In, Ic])
        J_shr = np.concatenate([Jn, Jc])
    else:
        I_shr = In
        J_shr = Jn

    vec1 = phi_test[neighs1] - phi_test[neighs0]
    vec2 = phi_test[neighs3] - phi_test[neighs2]

    if n_crn > 0:
        _grad1 = vec2[-n_crn:].flatten(order='F')
        _grad2 = vec1[-n_crn:].flatten(order='F')
        _grad0 = -_grad1 - _grad2

        grad1 = vec2[:-n_crn].flatten(order='F')
        grad0 = -grad1
        grad3 = vec1[:-n_crn].flatten(order='F')
        grad2 = -grad3
    else:
        grad1 = vec2.flatten(order='F')
        grad0 = -grad1
        grad3 = vec1.flatten(order='F')
        grad2 = -grad3
        _grad0 = np.array([])
        _grad1 = np.array([])
        _grad2 = np.array([])

    K_shr = np.concatenate([grad0, grad1, grad2, grad3, _grad0, _grad1, _grad2])

    grad_shr = csc_matrix((K_shr, (I_shr, J_shr)), shape=(n_conds_shr, 3*n_verts))
    print("grad_shr shape:", grad_shr.shape)
    print(grad_shr.toarray())
else:
    print("No shear constraints for this mesh.")

In [ ]:
# 32. Build one unconstrained dynamic step
dt = 0.0025
g = 9.8

rho = 0.1
delta = 0.1
alpha = 0.2
kappa = 0.5e-4
beta = 0.015 * kappa

D = alpha * M_lum + beta * K
K_bend = kappa * K
M_dyn = rho * M_lum

print("D =")
print(D.toarray())

print("\nK_bend =")
print(K_bend.toarray())

In [ ]:
M_dyn_xyz = sp.block_diag((M_dyn, M_dyn, M_dyn)).tocsc()
D_xyz = sp.block_diag((D, D, D)).tocsc()
K_bend_xyz = sp.block_diag((K_bend, K_bend, K_bend)).tocsc()

Fg_xyz = Fg.toarray().reshape((3*n_verts,), order='F')

print("M_dyn_xyz shape:", M_dyn_xyz.shape)
print("D_xyz shape:", D_xyz.shape)
print("K_bend_xyz shape:", K_bend_xyz.shape)

In [ ]:
# 33. Implicity Euler system matrix
E = M_dyn_xyz + dt * D_xyz + (dt**2) * K_bend_xyz
factor_E = cholesky(E)

# print("E shape:", E.shape)
# print(E.toarray())

In [ ]:
# 34. Compute the unconstrained step (\phi_0^{n+1})
phi_old = positions.reshape((3*n_verts,), order='F')
v_old = velocities.reshape((3*n_verts,), order='F')

q = (dt**2) * delta * g * Fg_xyz + dt * (M_dyn_xyz @ v_old) + (M_dyn_xyz + dt * D_xyz) @ phi_old
phi_unconstrained = factor_E(q)

print("phi_old:")
print(phi_old)

print("\nphi_unconstrained:")
print(phi_unconstrained)

# Reshape back to see it as xyz coordinates:
# This is the predicted motion before any stretch/shear collision correction.
phi_unconstrained_mat = phi_unconstrained.reshape((n_verts, 3), order='F')
print(phi_unconstrained_mat)

In [ ]:
# Exaggeration the downward motion due to gravity using a scale to visualize it
scale = 5000  # try 1000, 5000, 10000

phi_vis = positions + scale * (phi_unconstrained_mat - positions)
plot_mesh_compare(positions, phi_vis, edges_matrix, title=f"Unconstrained step (x{scale} visual exaggeration)")

In [ ]:
# 35. compare unconstrained vs original positions
# You should see the cloth moving downward due to gravity.
disp = phi_unconstrained_mat - positions

print("displacement:")
print(disp)

print("\nz displacement only:")
print(disp[:, 2])

In [ ]:
# 36. evaluate stretch and shear on the unconstrained step

# stretch
vec_uc = phi_unconstrained_mat[bars1] - phi_unconstrained_mat[bars0]
stretch_uc = np.einsum('ij,ij->i', vec_uc, vec_uc)
stretch_violation_uc = stretch_uc - stretch_val0

print("stretch_violation_uc:")
print(stretch_violation_uc)

# shear
if len(all_shear) > 0:
    vec1_uc = phi_unconstrained_mat[neighs1] - phi_unconstrained_mat[neighs0]
    vec2_uc = phi_unconstrained_mat[neighs3] - phi_unconstrained_mat[neighs2]
    shear_uc = np.einsum('ij,ij->i', vec1_uc, vec2_uc)
    shear_violation_uc = shear_uc - shear_val0

    print("\nshear_violation_uc:")
    print(shear_violation_uc)

In [ ]:
# 37. a very small projection demo for stretch only
# This is not the full simulator loop, but it helps you see the core idea.
# We solve one linearized projection step: \Delta \lambda = (A A^T + \beta I)^{-1}(-C)
# \phi \leftarrow \phi + M^{-1/2} A^T \Delta\lambda

phi_proj = phi_unconstrained.copy()

# rebuild stretch Jacobian at the current phi_proj
phi_proj_mat = phi_proj.reshape((n_verts, 3), order='F')
vec = phi_proj_mat[bars1] - phi_proj_mat[bars0]
val_str = np.einsum('ij,ij->i', vec, vec) - stretch_val0

grad1 = 2 * vec.flatten(order='F')
grad0 = -grad1
Kdata = np.concatenate([grad0, grad1])

I = np.tile(np.arange(n_conds_str), 6)
J = np.concatenate([
    bars0,
    bars0 + n_verts,
    bars0 + 2*n_verts,
    bars1,
    bars1 + n_verts,
    bars1 + 2*n_verts
])

A = csc_matrix((Kdata * m_sqrt_xyz[J], (I, J)), shape=(n_conds_str, 3*n_verts))
AT = A.T.tocsr()

par = 0.01
factor_AAT = cholesky_AAt(A, beta=par)

b = -val_str
dlam = factor_AAT(b)

phi_proj = phi_proj + m_sqrt_xyz * (AT @ dlam)
phi_proj_mat = phi_proj.reshape((n_verts, 3), order='F')

print("phi_proj_mat:")
print(phi_proj_mat)

In [ ]:
# 38. floor collision demo
phi_floor = phi_proj.copy()
phi_floor_mat = phi_floor.reshape((n_verts, 3), order='F').copy()

ind_col = np.nonzero(phi_floor_mat[:, 2] < 0)[0]
print("penetrating node indices:", ind_col)

phi_floor_mat[ind_col, 2] = 0.0

print(phi_floor_mat)

In [ ]:
# 39. self-collision candidate pairs using KDTree
# The KD-tree is a data structure for fast nearest-neighbor search in space
# For every node: look around locally in 3D space, collect nearby nodes,
# remove self-pairs, remove duplicate ordering, keep the pair distances.

kn = 12 if n_verts > 12 else n_verts - 1
nodes = np.arange(n_verts)
ni = np.repeat(np.arange(n_verts), kn)

tree_n = KDTree(phi_floor_mat)
dists, neighs = tree_n.query(phi_floor_mat, k=kn+1)

dist = dists[:, 1:].reshape(-1)
nj = neighs[:, 1:].reshape(-1)

mask = ni < nj
ni2 = ni[mask]
nj2 = nj[mask]
dist2 = dist[mask]

pairs = np.column_stack([ni2, nj2, dist2])

print("candidate pairs [i, j, distance]:")
print(pairs)

In [ ]:
# 40. The cloth ignores direct mesh-edge neighbors for self-collision.
share_edge = np.zeros((n_verts, n_verts), dtype=bool)
for edge in edges_matrix:
    a, b = edge
    share_edge[a, b] = True
    share_edge[b, a] = True

print(share_edge.astype(int))

# filter

mask_nonadj = ~share_edge[ni2, nj2]
pairs_nonadj = np.column_stack([ni2[mask_nonadj], nj2[mask_nonadj], dist2[mask_nonadj]])

print("non-adjacent close pairs:")
print(pairs_nonadj)

In [ ]:
# 41. package the most useful inspection into one function
def inspect_state(phi_mat, positions_rest, bars, shear_tuples=None):
    print("Current positions:\n", phi_mat)

    # stretch
    b0 = bars[:, 0]
    b1 = bars[:, 1]
    rest_vec = positions_rest[b1] - positions_rest[b0]
    now_vec = phi_mat[b1] - phi_mat[b0]

    rest_sq = np.einsum('ij,ij->i', rest_vec, rest_vec)
    now_sq = np.einsum('ij,ij->i', now_vec, now_vec)

    print("\nStretch violations:")
    print(now_sq - rest_sq)

    # shear
    if shear_tuples is not None and len(shear_tuples) > 0:
        n0 = shear_tuples[:, 0]
        n1 = shear_tuples[:, 1]
        n2 = shear_tuples[:, 2]
        n3 = shear_tuples[:, 3]

        rest_v1 = positions_rest[n1] - positions_rest[n0]
        rest_v2 = positions_rest[n3] - positions_rest[n2]
        now_v1 = phi_mat[n1] - phi_mat[n0]
        now_v2 = phi_mat[n3] - phi_mat[n2]

        rest_dot = np.einsum('ij,ij->i', rest_v1, rest_v2)
        now_dot = np.einsum('ij,ij->i', now_v1, now_v2)

        print("\nShear violations:")
        print(now_dot - rest_dot)

In [ ]:
inspect_state(phi_unconstrained_mat, positions, bars, all_shear)

In [ ]:
####### Animation

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [ ]:
def draw_mesh(ax, X, edges, color='tab:blue', linestyle='-', linewidth=2, label=None, show_points=True):
    for a, b in edges:
        ax.plot([X[a, 0], X[b, 0]],
                [X[a, 1], X[b, 1]],
                [X[a, 2], X[b, 2]],
                linestyle=linestyle, color=color, linewidth=linewidth)

    if show_points:
        ax.scatter(X[:, 0], X[:, 1], X[:, 2], color=color, s=40, label=label)

In [ ]:
def animate_mesh_history(history, edges_matrix, title="Mesh animation",
                         color='tab:red',
                         xlim=None, ylim=None, zlim=None,
                         interval=80,
                         show_reference=None,
                         reference_color='tab:blue'):
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    # automatic plot limits if not provided
    all_pts = np.vstack(history)
    if xlim is None:
        xlim = (all_pts[:, 0].min() - 0.2, all_pts[:, 0].max() + 0.2)
    if ylim is None:
        ylim = (all_pts[:, 1].min() - 0.2, all_pts[:, 1].max() + 0.2)
    if zlim is None:
        zlim = (all_pts[:, 2].min() - 0.2, all_pts[:, 2].max() + 0.2)

    def update(frame):
        ax.cla()

        X = history[frame]

        # optional reference mesh
        if show_reference is not None:
            draw_mesh(ax, show_reference, edges_matrix,
                      color=reference_color, linestyle='--', linewidth=1.2,
                      label='reference')

        # animated mesh
        draw_mesh(ax, X, edges_matrix,
                  color=color, linestyle='-', linewidth=2.0,
                  label='current')

        # ground plane z=0 for visual reference
        xx = np.linspace(xlim[0], xlim[1], 2)
        yy = np.linspace(ylim[0], ylim[1], 2)
        XX, YY = np.meshgrid(xx, yy)
        ZZ = np.zeros_like(XX)
        ax.plot_surface(XX, YY, ZZ, alpha=0.15)

        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.set_zlim(*zlim)

        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_zlabel("z")
        ax.set_title(f"{title} — frame {frame}")
        ax.view_init(elev=25, azim=-60)

    ani = FuncAnimation(fig, update, frames=len(history), interval=interval)
    plt.close(fig)
    return HTML(ani.to_jshtml())

Fall on the ground

In [ ]:
phi_tmp = positions.reshape((3*n_verts,), order='F').copy()
v_tmp = velocities.reshape((3*n_verts,), order='F').copy()

steps = 250
history_fall_ground = [positions.copy()]

for _ in range(steps):
    # unconstrained implicit step
    q = (dt**2) * delta * g * Fg_xyz + dt * (M_dyn_xyz @ v_tmp) + (M_dyn_xyz + dt * D_xyz) @ phi_tmp
    phi_next = factor_E(q)

    # reshape to (n_verts, 3)
    X_next = phi_next.reshape((n_verts, 3), order='F').copy()

    # floor collision: project penetrations to z=0
    X_next[:, 2] = np.maximum(X_next[:, 2], 0.0)

    # flatten back
    phi_next = X_next.reshape((3*n_verts,), order='F')

    # update velocity from displacement
    v_tmp = (phi_next - phi_tmp) / dt
    phi_tmp = phi_next

    history_fall_ground.append(X_next.copy())

In [ ]:
animate_mesh_history(
    history_fall_ground,
    edges_matrix,
    title="Falling and ground contact",
    color='tab:red',
    show_reference=positions,
    reference_color='tab:blue',
    interval=60
)

Shear deformation

In [ ]:
phi_shear_target = positions.copy()

# skew mainly in x-direction
phi_shear_target[4] += np.array([0.35, 0.00, 0.00])
phi_shear_target[3] += np.array([0.20, 0.00, 0.00])

n_frames = 60
history_shear = []

for t in np.linspace(0, 1, n_frames):
    X = (1 - t) * positions + t * phi_shear_target
    history_shear.append(X.copy())

In [ ]:
animate_mesh_history(
    history_shear,
    edges_matrix,
    title="Manual shear deformation",
    color='tab:green',
    show_reference=positions,
    reference_color='tab:blue',
    interval=80
)

Stretch animation

In [ ]:
phi_stretch_target = positions.copy()

# pull nodes in +x direction
phi_stretch_target[1] += np.array([0.30, 0.00, 0.00])
phi_stretch_target[4] += np.array([0.25, 0.00, 0.00])

n_frames = 60
history_stretch = []

for t in np.linspace(0, 1, n_frames):
    X = (1 - t) * positions + t * phi_stretch_target
    history_stretch.append(X.copy())

In [ ]:
animate_mesh_history(
    history_stretch,
    edges_matrix,
    title="Manual stretch deformation",
    color='tab:orange',
    show_reference=positions,
    reference_color='tab:blue',
    interval=80
)

Combined